# Solution F1 · Gini or entropy? Best split or random split?

**Type**: Compare

## Objective

Systematically compare the impurity criterion (`gini`, `entropy`) and the `splitter` (`best`, `random`) of a tree on three datasets (wines, tumors, digits) with a repeated cross-validation, measuring accuracy, number of leaves and training time, and establish how much these choices matter compared with the measurement noise
of a single tree.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

import time
from sklearn.datasets import load_breast_cancer, load_digits, load_wine
from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate
from sklearn.tree import DecisionTreeClassifier

datasets = {"wine": load_wine(return_X_y=True), "tumors": load_breast_cancer(return_X_y=True), "digits": load_digits(return_X_y=True)}
combinations = {"gini/best": ("gini", "best"), "entropy/best": ("entropy", "best"), "gini/random": ("gini", "random"), "entropy/random": ("entropy", "random")}
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=4, random_state=0)

## Commented solution

In [ ]:
rows, leaves = {}, {}
for name, (X, y) in datasets.items():
    for comb, (criterion, splitter) in combinations.items():
        model = DecisionTreeClassifier(criterion=criterion, splitter=splitter, min_samples_leaf=3, random_state=0)
        r = cross_validate(model, X, y, cv=cv)
        rows[(name, comb)] = {"acc": r["test_score"].mean(), "sd": r["test_score"].std(), "t_fit": r["fit_time"].mean()}
        leaves[(name, comb)] = model.fit(X, y).get_n_leaves()
table = pd.DataFrame(rows).T
leaves = pd.Series(leaves)
table.assign(leaves=leaves).round(3)

In [ ]:
criterion_diff = {name: table.loc[(name, "entropy/best"), "acc"] - table.loc[(name, "gini/best"), "acc"] for name in datasets}
max_criterion_diff = max(abs(v) for v in criterion_diff.values())
random_has_more_leaves = bool(all(leaves[(name, "gini/random")] > leaves[(name, "gini/best")] for name in datasets))
typical_sd = float(table["sd"].median())
criterion_matters = bool(max_criterion_diff >= 2 * typical_sd)
print("acc(entropy) - acc(gini):", {k: round(v, 3) for k, v in criterion_diff.items()}, f"| maximum {max_criterion_diff:.3f}")
print(f"typical standard deviation across folds: {typical_sd:.3f} | the criterion matters in a distinguishable way: {criterion_matters} | random has more leaves everywhere: {random_has_more_leaves}")

## Check

In [ ]:
with verify():
    check(table.shape == (12, 3) and list(table.columns) == ["acc", "sd", "t_fit"] and len(leaves) == 12, "12 rows (3 datasets x 4 combinations) and three columns; 12 leaf counts", "acc, sd, t_fit")
    check(set(criterion_diff) == {"wine", "tumors", "digits"} and close(max_criterion_diff, max(abs(v) for v in criterion_diff.values())) and close(criterion_diff["wine"], table.loc[("wine", "entropy/best"), "acc"] - table.loc[("wine", "gini/best"), "acc"]), "differences between criteria computed", "acc(entropy/best) - acc(gini/best)")
    check(max_criterion_diff < 0.04, "the difference between the two criteria is always below 4 points", "compare the acc values")
    check(random_has_more_leaves is True and (leaves[[("wine", "gini/random"), ("tumors", "gini/random"), ("digits", "gini/random")]].to_numpy() > leaves[[("wine", "gini/best"), ("tumors", "gini/best"), ("digits", "gini/best")]].to_numpy()).all(), "the random splitter gives more leaves, on all the datasets", "leaves of gini/random against gini/best")
    check(0.015 < typical_sd < 0.06 and criterion_matters is False, "the standard deviation across folds (2-6 points) is of the same order as the differences: the criterion does not matter in a distinguishable way", "median of the sd")
done()

## What you learned

- **Gini against entropy: differences of 1-3 points, of the same order as the noise.** The standard deviation across the folds of a single tree is 2-5 points: a difference of one point cannot be distinguished. Both criteria are concave functions of the impurity with a maximum at 0.5: they almost always choose the same split. Neither is "better": try both if you have time and do not
  expect miracles.
- **`splitter="random"` does not worsen the accuracy** but produces **bigger** trees (about 1.3-1.7 times the leaves) because it picks a random threshold instead of the best one for each feature. It costs less per node (no exhaustive search) and it is used (with other tricks) in "extremely randomized trees".
- **The real point is another:** regularization (minimum leaf size, depth, pruning) moves the accuracy by 10 points (B1), the criterion by 1-3. Optimizing the criterion before the regularization is a way of wasting time. And, with a cross-validation of only 5 folds, differences of that kind would sometimes go one way and sometimes the other.
- **The time** is almost the same between Gini and entropy (the entropy needs one more logarithm), and with `random` it is lower on datasets with many features (digits).

## Common pitfall

Reporting "entropy is better than Gini" from a single cross-validation with a difference of one point. A tree is an unstable model: without repeating the validation and without looking at the variability between folds there is no way to tell a real effect from noise.

## Interview question

<details><summary>Which criterion do you use to split a tree: Gini or entropy?</summary>

In practice they give almost identical trees: I pick the default (Gini, slightly faster) and focus on what really matters, which is the regularization (minimum leaf size, depth, pruning) chosen in cross-validation. If I want to compare the two criteria I do it with a repeated validation and looking at the variability.

</details>